# Experiment 10a - Feature engineering: combine, interact, dynamicize

Phase 9 established the **headline finding**: across tree-GBMs (r=0.9955) and a
fine-tuned foundation model (r=0.90), every covariate model converges to the same
function - the ceiling is the **feature set, not the model class**. So the only lever
left that can move the floor is **new information in the features**.

This notebook tests three feature-engineering blocks as an **additive ablation** on
the LGBM-MAE testbed (fast, CPU; ports to the MLP if a block wins), 3-fold
rolling-origin CV, each gated against the **0.05 WAPE floor** vs the existing 52-feature
baseline.

**The guiding principle (from the literature + our own convergence finding):**
A model that can represent linear combinations (GBM, MLP) gains ~nothing from a
*linear* recombination of correlated inputs - it already computes those internally
(GBMs are provably immune to multicollinearity). So the blocks are ordered by how much
**new** information they add, not by intuition:

| Block | What | Adds new info? | Honest expectation |
|---|---|---|---|
| **A. Stress factor** | PLS (supervised) + PCA of the 5 collinear covariates -> 4 composite features | No (linear reorg) | ~flat for LGBM/MLP; interpretable; may help a foundation model |
| **B. Interactions** | products/ratios (e.g. `queue x shock`, `workload / reliability`) | **Yes** (non-additive) | a real hope - trees need deep splits to approximate products |
| **C. Temporal dynamics** | velocity (Delta), Delta-24, EWMA, regime flag | **Yes** (absent from instantaneous values) | the other real hope; also targets the universally-hard fold 1 |

**Pre-registered rule:** keep a block only if it beats baseline by > 0.05 CV-WAPE on
all-fold mean, watching fold-1 specifically. If even B and C don't move it, that
confirms the covariates' information is fully extracted and ~14.5 is the irreducible
floor for this feature set (only *external* new data could help then).


---
## §0. Configuration

In [3]:
from pathlib import Path

DATA_DIR   = Path("/kaggle/input/datasets/xitij27/dlam-bonus-project-dataset/data")
OUTPUT_DIR = Path("/kaggle/working")
# Local:
# DATA_DIR   = Path("data")
# OUTPUT_DIR = Path("experiment-10")

TRAIN_PATH = DATA_DIR / "train.csv"
SEED, N_FOLDS, EVAL_HORIZON = 42, 3, 336

DOMAIN_FEAT_COLS = [
    "workload_intensity", "demand_forecast", "upstream_quality_forecast",
    "promotion_intensity", "shock_risk", "maintenance_known",
    "unit_reliability_forecast", "queue_pressure_forecast",
    "network_pressure_forecast", "event_load_forecast",
    "service_irregularity_risk_forecast",
]
TIME_COLS = ["hour_sin","hour_cos","dow_sin","dow_cos","doy_sin","doy_cos","is_weekend"]
TOP_DOM   = ["queue_pressure_forecast","network_pressure_forecast","workload_intensity","shock_risk","event_load_forecast"]
TOP3_ROLL = TOP_DOM[:3]
LAGS_DOM, LEADS_DOM, ROLL_WINS = [1,24], [1,24], [6,24]

# the 5 collinear "operational stress" covariates for the factor block
CORR_COLS = ["queue_pressure_forecast","network_pressure_forecast","workload_intensity","shock_risk","event_load_forecast"]
DYN_BASE  = ["queue_pressure_forecast","network_pressure_forecast","workload_intensity"]

BASELINE_LGBM = dict(n_estimators=600, learning_rate=0.03, num_leaves=63,
                     subsample=0.8, colsample_bytree=0.8, min_child_samples=50,
                     reg_lambda=1.0, verbose=-1)
SEEDS = [42, 101, 202]
ANCHOR_WAPE, ACCEPT_DELTA = 14.78, 0.05

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"blocks: A=stress factor (PLS+PCA), B=interactions, C=temporal dynamics")
print(f"accept a block iff CV-WAPE beats baseline by > {ACCEPT_DELTA}")


blocks: A=stress factor (PLS+PCA), B=interactions, C=temporal dynamics
accept a block iff CV-WAPE beats baseline by > 0.05


---
## §1. Setup + load + impute

In [4]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
np.random.seed(SEED)

def wape(y, p, eps=1e-8):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return np.sum(np.abs(y - p)) / (np.sum(np.abs(y)) + eps) * 100

train_df = pd.read_csv(TRAIN_PATH, parse_dates=["timestamp"])
sid_to_code = {s:i for i,s in enumerate(sorted(train_df.series_id.unique()))}
N_SERIES = len(sid_to_code)
print(f"train_df {train_df.shape}, series {N_SERIES}")


train_df (414720, 25), series 96


---
## §2. Baseline 52-feature pipeline (unchanged from exp-5.1/9)

In [5]:
def impute(df, cols):
    df = df.sort_values(["series_id","timestamp"]).copy()
    df[cols] = df.groupby("series_id", group_keys=False)[cols].transform(lambda x: x.ffill().bfill())
    return df
def add_time(df):
    df = df.copy(); t = pd.to_datetime(df.timestamp)
    df["hour_sin"]=np.sin(2*np.pi*t.dt.hour/24); df["hour_cos"]=np.cos(2*np.pi*t.dt.hour/24)
    df["dow_sin"]=np.sin(2*np.pi*t.dt.dayofweek/7); df["dow_cos"]=np.cos(2*np.pi*t.dt.dayofweek/7)
    df["doy_sin"]=np.sin(2*np.pi*t.dt.dayofyear/365); df["doy_cos"]=np.cos(2*np.pi*t.dt.dayofyear/365)
    df["is_weekend"]=(t.dt.dayofweek>=5).astype(float); return df
def add_temporal(df, lag_cols, roll_cols, lags, leads, wins):
    df = df.copy().sort_values(["series_id","timestamp"]); new=[]
    for c in lag_cols:
        g=df.groupby("series_id",sort=False)[c]
        for L in lags: k=f"{c}_lag{L}"; df[k]=g.shift(L); new.append(k)
        for L in leads: k=f"{c}_lead{L}"; df[k]=g.shift(-L); new.append(k)
    for c in roll_cols:
        g=df.groupby("series_id",sort=False)[c]
        for W in wins:
            km=f"{c}_roll{W}m"; df[km]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).mean()); new.append(km)
            ks=f"{c}_roll{W}s"; df[ks]=g.shift(1).transform(lambda x:x.rolling(W,min_periods=1).std().fillna(0)); new.append(ks)
    return df, new

base = add_time(impute(train_df, DOMAIN_FEAT_COLS))
base["sid_code"] = base.series_id.map(sid_to_code).astype("int32")
base, TEMPORAL_FEATS = add_temporal(base, TOP_DOM, TOP3_ROLL, LAGS_DOM, LEADS_DOM, ROLL_WINS)
FEATS_BASE = DOMAIN_FEAT_COLS + TIME_COLS + ["series_mean"] + TEMPORAL_FEATS + ["sid_code"]
print(f"baseline features: {len(FEATS_BASE)}")


baseline features: 52


---
## §3. Block B (interactions) + Block C (dynamics) - deterministic, added to `base`

Both are causal/pointwise (no fitting, no future leak): interactions are per-row products/ratios; dynamics use backward `shift`/`diff`. Block A (PLS/PCA) is fit **per fold** in §4 because PLS uses the target.

In [6]:
# --- Block B: interactions (non-additive signal) ---
INT_COLS = []
def add_int(name, series):
    base[name] = series; INT_COLS.append(name)
add_int("int_queue_shock",   base.queue_pressure_forecast   * base.shock_risk)
add_int("int_network_shock", base.network_pressure_forecast * base.shock_risk)
add_int("int_queue_network", base.queue_pressure_forecast   * base.network_pressure_forecast)
add_int("int_workload_shock",base.workload_intensity        * base.shock_risk)
add_int("int_queue_event",   base.queue_pressure_forecast   * base.event_load_forecast)
add_int("ratio_queue_reliab",   base.queue_pressure_forecast / (base.unit_reliability_forecast.abs() + 1.0))
add_int("ratio_workload_quality",base.workload_intensity     / (base.upstream_quality_forecast.abs() + 1.0))

# --- Block C: temporal dynamics (causal) ---
DYN_COLS = []
for col in DYN_BASE:
    g = base.groupby("series_id", sort=False)[col]
    base[f"{col}_vel1"]  = g.diff(1);  DYN_COLS.append(f"{col}_vel1")
    base[f"{col}_vel24"] = g.diff(24); DYN_COLS.append(f"{col}_vel24")
    base[f"{col}_ewm12"] = g.transform(lambda x: x.shift(1).ewm(span=12, min_periods=1).mean()); DYN_COLS.append(f"{col}_ewm12")
gq = base.groupby("series_id", sort=False)["queue_pressure_forecast"]
roll24 = gq.transform(lambda x: x.shift(1).rolling(24, min_periods=1).mean())
expmed = gq.transform(lambda x: x.shift(1).expanding(min_periods=1).median())
base["stress_regime"] = (roll24 > expmed).astype(float); DYN_COLS.append("stress_regime")

print(f"Block B interactions: {len(INT_COLS)} -> {INT_COLS}")
print(f"Block C dynamics:     {len(DYN_COLS)} -> {DYN_COLS}")


Block B interactions: 7 -> ['int_queue_shock', 'int_network_shock', 'int_queue_network', 'int_workload_shock', 'int_queue_event', 'ratio_queue_reliab', 'ratio_workload_quality']
Block C dynamics:     10 -> ['queue_pressure_forecast_vel1', 'queue_pressure_forecast_vel24', 'queue_pressure_forecast_ewm12', 'network_pressure_forecast_vel1', 'network_pressure_forecast_vel24', 'network_pressure_forecast_ewm12', 'workload_intensity_vel1', 'workload_intensity_vel24', 'workload_intensity_ewm12', 'stress_regime']


---
## §4. CV framework + Block A (per-fold PLS/PCA) + LGBM fit

In [7]:
def fold_offset(k): return (N_FOLDS - k) * EVAL_HORIZON
def fold_split(end_off, length=EVAL_HORIZON):
    tr_l, te_l = [], []
    for _, g in base.groupby("series_id", sort=False):
        g = g.sort_values("timestamp"); cut = len(g) - end_off
        tr_l.append(g.iloc[:cut]); te_l.append(g.iloc[cut:cut+length])
    tr = pd.concat(tr_l, ignore_index=True); te = pd.concat(te_l, ignore_index=True)
    sm = tr.groupby("series_id")["target"].mean().rename("series_mean").reset_index()
    return tr.merge(sm, on="series_id"), te.merge(sm, on="series_id")

def add_stress_factor(tr, te):
    """Block A: fit PLS (supervised) + PCA on the fold's TRAIN only; transform both."""
    sc = StandardScaler().fit(tr[CORR_COLS])
    Xtr, Xte = sc.transform(tr[CORR_COLS]), sc.transform(te[CORR_COLS])
    pls = PLSRegression(n_components=2).fit(Xtr, tr["target"].to_numpy(float))
    pca = PCA(n_components=2, random_state=SEED).fit(Xtr)
    tr, te = tr.copy(), te.copy(); cols = ["stress_pls1","stress_pls2","stress_pca1","stress_pca2"]
    tr[["stress_pls1","stress_pls2"]] = pls.transform(Xtr); te[["stress_pls1","stress_pls2"]] = pls.transform(Xte)
    tr[["stress_pca1","stress_pca2"]] = pca.transform(Xtr); te[["stress_pca1","stress_pca2"]] = pca.transform(Xte)
    return tr, te, cols

def fit_lgbm(tr, te, feats):
    preds = []
    for s in SEEDS:
        m = lgb.LGBMRegressor(**BASELINE_LGBM, objective="regression_l1", random_state=s)
        m.fit(tr[feats], tr["target"].to_numpy(float), categorical_feature=["sid_code"])
        preds.append(m.predict(te[feats]))
    return np.clip(np.mean(preds, axis=0), 0, None)

print("CV framework + Block A ready.")


CV framework + Block A ready.


---
## §5. Additive ablation - run all configs on the 3-fold CV

In [8]:
CONFIGS = [
    ("baseline",        [],                       False),
    ("+A stress factor",[],                       True),
    ("+B interactions", INT_COLS,                 False),
    ("+C dynamics",     DYN_COLS,                 False),
    ("+B +C",           INT_COLS + DYN_COLS,      False),
    ("+all (A+B+C)",    INT_COLS + DYN_COLS,      True),
]
results = {}
for name, extra, use_stress in CONFIGS:
    per_fold = []
    for k in range(N_FOLDS):
        tr, te = fold_split(fold_offset(k))
        feats = FEATS_BASE + list(extra)
        if use_stress:
            tr, te, scols = add_stress_factor(tr, te); feats = feats + scols
        per_fold.append(wape(te["target"], fit_lgbm(tr, te, feats)))
    results[name] = per_fold
    print(f"  {name:<18}: folds {[round(x,2) for x in per_fold]}  mean {np.mean(per_fold):.3f}")

base_mean = np.mean(results["baseline"])
print(f"\n{'config':<18} {'fold0':>7} {'fold1':>7} {'fold2':>7} {'mean':>7} {'Δ vs base':>10}")
print("-"*62)
for name, pf in results.items():
    m = np.mean(pf); d = base_mean - m
    flag = "" if name=="baseline" else ("  ACCEPT" if d>ACCEPT_DELTA else ("  ~noise" if d>-ACCEPT_DELTA else "  worse"))
    print(f"{name:<18} {pf[0]:>7.3f} {pf[1]:>7.3f} {pf[2]:>7.3f} {m:>7.3f} {d:>+10.3f}{flag}")


  baseline          : folds [np.float64(14.05), np.float64(15.55), np.float64(14.62)]  mean 14.738
  +A stress factor  : folds [np.float64(14.07), np.float64(15.58), np.float64(14.74)]  mean 14.795
  +B interactions   : folds [np.float64(14.07), np.float64(15.57), np.float64(14.67)]  mean 14.769
  +C dynamics       : folds [np.float64(14.01), np.float64(15.53), np.float64(14.56)]  mean 14.700
  +B +C             : folds [np.float64(14.02), np.float64(15.53), np.float64(14.62)]  mean 14.723
  +all (A+B+C)      : folds [np.float64(14.04), np.float64(15.51), np.float64(14.69)]  mean 14.749

config               fold0   fold1   fold2    mean  Δ vs base
--------------------------------------------------------------
baseline            14.045  15.549  14.619  14.738     +0.000
+A stress factor    14.066  15.576  14.744  14.795     -0.057  worse
+B interactions     14.072  15.567  14.667  14.769     -0.031  ~noise
+C dynamics         14.010  15.533  14.557  14.700     +0.038  ~noise
+B +C    

---
## §6. Verdict + save

In [9]:
import pandas as pd
rows = [{"config":n, "fold0":pf[0], "fold1":pf[1], "fold2":pf[2],
         "mean":float(np.mean(pf)), "delta_vs_base":float(base_mean-np.mean(pf))}
        for n, pf in results.items()]
summ = pd.DataFrame(rows)
summ.to_csv(OUTPUT_DIR/"ablation_10a_features.csv", index=False)
print(f"summary -> {OUTPUT_DIR/'ablation_10a_features.csv'}")

winners = [n for n,pf in results.items() if n!="baseline" and (base_mean-np.mean(pf))>ACCEPT_DELTA]
print("\n=== Verdict (Frontier 4 - feature engineering) ===")
print(f"  baseline CV-WAPE: {base_mean:.3f}")
if winners:
    best = min(winners, key=lambda n: np.mean(results[n]))
    print(f"  WINNERS (> {ACCEPT_DELTA}): {winners}")
    print(f"  best: {best} @ {np.mean(results[best]):.3f} ({base_mean-np.mean(results[best]):+.3f})")
    print("  -> port the winning block(s) to the gated-MLP (the 14.59 model) and the production pipeline.")
    print("  -> check fold-1 specifically: did the gain come proportionally, or only off-fold-1?")
else:
    a = base_mean - np.mean(results["+A stress factor"])
    bc = base_mean - np.mean(results["+B +C"])
    print(f"  No block clears {ACCEPT_DELTA}. A:{a:+.3f}  B+C:{bc:+.3f}")
    print("  Interpretation: if even nonlinear interactions + temporal dynamics of the existing")
    print("  covariates don't move it, the covariates' information is fully extracted ->")
    print("  ~14.5 is the irreducible floor for THIS feature set. Only external new data could help.")
    print("  This is itself a clean, defensible result for the writeup (confirms the Phase-9 finding).")


summary -> /kaggle/working/ablation_10a_features.csv

=== Verdict (Frontier 4 - feature engineering) ===
  baseline CV-WAPE: 14.738
  No block clears 0.05. A:-0.057  B+C:+0.015
  Interpretation: if even nonlinear interactions + temporal dynamics of the existing
  covariates don't move it, the covariates' information is fully extracted ->
  ~14.5 is the irreducible floor for THIS feature set. Only external new data could help.
  This is itself a clean, defensible result for the writeup (confirms the Phase-9 finding).
